# Statewide TIGER/Line build

This is a separate whole-state example. It downloads all TIGER/Line address-range files for Delaware by calling `download_tiger_ranges(state="DE", county=None)`, prepares the ranges with Delaware State Plane coordinates, stores the result locally, and geocodes five addresses from the prepared statewide table.

Delaware keeps this runnable as a checked-in demo. For a larger state, the same call works, but the download, interpolation, and local DuckDB files will be larger. The input data never goes to an online geocoder.

In [1]:
from pathlib import Path
from time import perf_counter

import pandas as pd

from geotiger import (
    GeoTIGERStore,
    Geocoder,
    GeocoderConfig,
    InterpolationConfig,
    download_tiger_ranges,
    load_ranges,
    prepare_ranges,
    state_plane_crs,
)

STATE = "DE"
YEAR = 2024
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
CACHE_DIR = PROJECT_ROOT / "data/statewide_demo"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
RANGES_CACHE = CACHE_DIR / "delaware_tiger_ranges.parquet"
PREPARED_CACHE = CACHE_DIR / "delaware_tiger_prepared.parquet"
DATABASE = CACHE_DIR / "delaware_tiger.duckdb"

print({"state": STATE, "county": None, "year": YEAR, "projection": state_plane_crs(STATE)})

{'state': 'DE', 'county': None, 'year': 2024, 'projection': 'EPSG:2243'}


## 1. Download and cache the whole state

`county=None` is the important part: GeoTIGER obtains the state’s county list and downloads every county. After this cell has run once, subsequent runs use the local Parquet cache.

In [2]:
download_started = perf_counter()
if RANGES_CACHE.exists():
    ranges = load_ranges(RANGES_CACHE)
    download_mode = "loaded local cache"
else:
    ranges = download_tiger_ranges(STATE, county=None, year=YEAR, cache=True)
    ranges.to_parquet(RANGES_CACHE)
    download_mode = "downloaded and cached"
download_seconds = perf_counter() - download_started

print({
    "mode": download_mode,
    "range_rows": len(ranges),
    "source_crs": ranges.crs.to_epsg() if ranges.crs is not None else None,
    "seconds_this_run": round(download_seconds, 3),
})

{'mode': 'loaded local cache', 'range_rows': 62067, 'source_crs': 4269, 'seconds_this_run': 0.126}


## 2. Prepare statewide interpolation points

The interpolation is done in Delaware State Plane (`EPSG:2243`), not latitude/longitude. Latitude/longitude are retained in the prepared output for geocoding results. The zero offsets below make the demo points land on the street centerline.

The higher range limit is intentional for statewide TIGER data: one Delaware source range expands to 130,551 potential numbers.

In [3]:
prep_config = InterpolationConfig(
    projected_crs=state_plane_crs(STATE),
    end_offset_m=0.0,
    side_offset_m=0.0,
    max_addresses_per_range=150_000,
    include_intersections=True,
)

prep_started = perf_counter()
if PREPARED_CACHE.exists():
    prepared = pd.read_parquet(PREPARED_CACHE)
    prep_mode = "loaded local cache"
else:
    prepared = prepare_ranges(
        ranges,
        state=STATE,
        source=f"tiger_{YEAR}_{STATE.lower()}",
        config=prep_config,
    )
    prepared.to_parquet(PREPARED_CACHE)
    prep_mode = "prepared and cached"
prep_seconds = perf_counter() - prep_started

print({
    "mode": prep_mode,
    "prepared_rows": len(prepared),
    "intersection_rows": int(prepared["is_intersection"].sum()),
    "interpolation_crs": prepared["interpolation_crs"].iloc[0],
    "seconds_this_run": round(prep_seconds, 3),
})

{'mode': 'loaded local cache', 'prepared_rows': 5465835, 'intersection_rows': 29963, 'interpolation_crs': 'EPSG:2243', 'seconds_this_run': 1.047}


## 3. Build a local statewide store and geocode

The DuckDB file is also local. The sample addresses are generated from the prepared data itself, so this cell checks the complete path without relying on an online service or a second data source.

In [4]:
sample = (
    prepared.loc[
        (~prepared["is_intersection"])
        & prepared["house_number"].notna()
        & prepared["street_norm"].ne("")
    ]
    .drop_duplicates(["house_number", "street_norm", "zip5"])
    .head(5)
    .copy()
)
inputs = pd.DataFrame({
    "address": sample["house_number"].astype(int).astype(str) + " " + sample["street_norm"],
    "state": sample["state_norm"],
    "zip": sample["zip5"],
})

store_started = perf_counter()
with GeoTIGERStore(DATABASE, threads=4) as store:
    store.ingest_candidates(prepared, replace=True)
    ingest_seconds = perf_counter() - store_started
    result = Geocoder(
        store,
        config=GeocoderConfig(
            strict_locality=False,
            street_fallback=False,
            deduplicate_inputs=True,
        ),
    ).geocode(inputs)
    store_counts = {
        "address_rows": store.count(),
        "intersection_rows": store.intersection_count(),
    }

print({
    **store_counts,
    "ingest_seconds": round(ingest_seconds, 3),
    "matched": int((result.matches["match_status"] == "matched").sum()),
    "unmatched": int((result.matches["match_status"] == "unmatched").sum()),
    "geocode_seconds": result.timings.total_seconds,
})
result.matches[["address", "match_status", "matched_address_id", "match_latitude", "match_longitude"]]

{'address_rows': 5465835, 'intersection_rows': 29963, 'ingest_seconds': 78.507, 'matched': 5, 'unmatched': 0, 'geocode_seconds': 0.20602349999535363}


,address,match_status,matched_address_id,match_latitude,match_longitude
0,1798 DIXIELINE RD,matched,tiger_2024_de:187290678:L:1798,39.632877,-75.787276
1,1796 DIXIELINE RD,matched,tiger_2024_de:187290678:L:1796,39.632837,-75.787273
2,1794 DIXIELINE RD,matched,tiger_2024_de:187290678:L:1794,39.632796,-75.787270
3,1792 DIXIELINE RD,matched,tiger_2024_de:187290678:L:1792,39.632756,-75.787267
4,1790 DIXIELINE RD,matched,tiger_2024_de:187290678:L:1790,39.632716,-75.787263


## Scaling to another state

Change only `STATE` and the cache names. For example, `download_tiger_ranges("NC", county=None, year=2024, cache=True)` follows the same all-counties path for North Carolina. Keep the range and prepared caches outside Git for large states; the notebook is intentionally the small, reviewable artifact.